# E07 Kaggle CUDA and saved-output storage qualification
E07 seed42 smoke only. Zero research updates. Frozen scientific source and exact approved software profile. Publish/save and independently recover before research.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT=Path('/tmp/e07_source'); ORCH=Path('/tmp/e07_orchestration')
OUT=Path('/kaggle/working')
SOURCE='5bb03b18943e7c45a305ce1d78b3446c31829ae7'
ORCH_COMMIT='942b1efcf859560019ccc6a0e5adeb9ca15e9e5c'
env={**os.environ,'UV_PYTHON_INSTALL_DIR':'/tmp/e07_python','UV_CACHE_DIR':'/tmp/e07_cache','UV_NATIVE_TLS':'true','CUBLAS_WORKSPACE_CONFIG':':4096:8','OMP_NUM_THREADS':'1','MPLCONFIGDIR':'/tmp/e07_mpl','MPLBACKEND':'Agg','PYTHONPATH':str(ORCH)}
def command(args,cwd=None):
    print('EXEC',args,flush=True)
    with (OUT/'bootstrap.log').open('a') as log:
        result=subprocess.run(args,cwd=cwd,env=env,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((OUT/'bootstrap.log').read_text()[-4000:],flush=True)
        raise RuntimeError('Bootstrap or qualification failed; preserve output')
assert not ROOT.exists() and not ORCH.exists()
for destination,commit in [(ROOT,SOURCE),(ORCH,ORCH_COMMIT)]:
    command(['git','clone','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(destination)])
    command(['git','checkout','--detach',commit],destination)
    assert not subprocess.check_output(['git','status','--porcelain'],cwd=destination,text=True).strip()
command([sys.executable,'-m','pip','install','--target','/tmp/e07_tools','uv==0.12.7'])
UV=Path('/tmp/e07_tools/bin/uv')
command([str(UV),'python','install','3.11.14'])
command([str(UV),'venv','--seed','--python','3.11.14','/tmp/e07_venv'])
PY=Path('/tmp/e07_venv/bin/python')
freeze=(ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock=OUT/'qualified-lock.txt';lock.write_text(chr(10).join(x for x in freeze if not x.startswith('-e '))+chr(10))
command([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'])
command([str(UV),'pip','install','--python',str(PY),'-r',str(lock)])
command([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'])
command([str(PY),'-m','pip','check'])
command([str(PY),str(ORCH/'scripts/pilot/e07_kaggle.py'),'preflight','--root',str(ROOT),'--output',str(OUT/'qualification')],ROOT)
print((OUT/'qualification/preflight.json').read_text(),flush=True)
print((OUT/'E07_smoke_manifest.json').read_text(),flush=True)
print('QUALIFICATION COMPLETE; RESEARCH UPDATES: 0',flush=True)
